In [ ]:
import pandas as pd

from gamblersruin.data import fetch_history
from gamblersruin.odds import LEAGUES, fetch_matches, kickoff_uk_naive, to_understat
from gamblersruin.predictor import MatchPredictor, with_ev

SEASONS = ['2025-2026', '2026-2027']  # last season, this season
DAY = None

In [ ]:
matches = fetch_matches(LEAGUES, DAY)

if matches.empty:
    print("No matches")
matches

In [ ]:
predictors = {}

for competition_id in matches.competition_id.unique():
    league = LEAGUES[competition_id]
    history = fetch_history(league, SEASONS)
    predictors[competition_id] = MatchPredictor(history, SEASONS)
    print(league, len(history), "matches", history.groupby("season").size().to_dict())

In [ ]:
unmapped = {}

for match in matches.itertuples():
    predictor = predictors[match.competition_id]
    home, away = to_understat(match.home), to_understat(match.away)
    unknown = [nt for nt, team in ((match.home, home), (match.away, away)) if team not in predictor.known_teams]
    if unknown:
        unmapped.setdefault(match.competition_id, set()).update(unknown)
        print(f"Skipping {match.home} vs {match.away} ({match.competition}): {unknown} not in matches\n")
        continue
    odds = (match.odds_home, match.odds_draw, match.odds_away)
    odds = None if any(pd.isna(o) for o in odds) else odds
    print(f"== {match.competition} ==")
    for warning in predictor.promoted_warnings(home, away):
        print(warning)
    prediction = predictor.predict(home, away, kickoff_uk_naive(match.kickoff), odds)
    print(prediction.format())
    display(prediction.table if odds is None else with_ev(prediction.table, odds))
    print()

for competition_id, names in unmapped.items():
    print(f"Add these to NT_TO_UNDERSTAT in gamblersruin/odds.py (Understat names for {LEAGUES[competition_id]} are in "
          f"predictors['{competition_id}'].known_teams):")
    for name in sorted(names):
        print(f'    "{name}": "",')